# Validate transport into a new cohort

Given a frozen reference and an independent query cohort, `validate_transport` decides for
every reference state whether it carries over well enough to enter downstream analyses. It
does not map cells: the mapping comes from the reference's own kNN baseline or from any other
tool through `Predictions`.

In [1]:
import tempfile
import warnings
from pathlib import Path

import pandas as pd

import statetrace as st
from statetrace.datasets import simulate_transport

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 20)
workdir = Path(tempfile.mkdtemp(prefix="statetrace_tutorial_"))
# show warnings as one line, without the source file
warnings.formatwarning = (
    lambda message, category, *args, **kwargs: f"{category.__name__}: {message}\n"
)
st.__version__

'0.1.0'

In [2]:
ref_adata, query_adata, truth = simulate_transport(
    n_ref_groups=8,
    n_query_groups=6,
    n_states=5,
    n_latent=10,
    n_genes=300,
    shifted_states={"S1": 9.0},
    absent_states=["S4"],
    minority_states={"S3": 0.25},
    seed=5,
)
ref_adata

AnnData object with n_obs × n_vars = 1600 × 300
    obs: 'donor_id', 'cell_state', 'cell_type', 'batch', 'study'
    uns: 'statetrace_synthetic'
    obsm: 'X_latent'
    layers: None (.X)

In [3]:
ref = st.Reference(
    ref_adata, group_key="donor_id", state_key="cell_state", latent_key="X_latent"
).fit()
ref.leave_one_group_out()
ref = st.Reference.load(ref.freeze(workdir / "reference.statetrace")).fit()

## Transport

The query must not share donors with the reference. Its embedding sits in `obsm["X_latent"]`,
the same space as the reference, which enables the distance-based novelty diagnostics.

In [4]:
route = st.validate_transport(ref, query_adata, group_key="donor_id")
print(route.summary())

StateTrace transport (knn) over 'donor_id': 1200 query cells, 6 groups, 5 reference states
  status: supported=2, supported_with_caution=0, shifted=1, insufficient=1, absent=1
  cells: supported 0.755 | uncertain 0.000 | out_of_reference 0.245


In [5]:
route.states[
    [
        "ref_reproducibility",
        "groups_recovered",
        "query_groups",
        "shift_level",
        "shift_source",
        "ood_fraction",
        "status",
    ]
].join(truth["truth"])

,ref_reproducibility,groups_recovered,query_groups,shift_level,shift_source,ood_fraction,status,truth
state,,,,,,,,
S0,stable,6,6,low,expression,0.000000,supported,stable
S1,stable,0,6,high,expression,0.979661,shifted,shifted
S2,stable,6,6,low,expression,0.017182,supported,stable
S3,insufficient,6,6,low,latent,0.000000,insufficient,minority
S4,stable,0,6,unknown,none,NaN,absent,absent


Every planted scenario receives its status:

- `S0`, `S2` (stable) are **supported**: reproducible in the reference, recovered in every query
  donor, not shifted.
- `S1` (shifted) is **shifted**: its query cells sit far from the reference state, in the latent
  (`ood_fraction`) and in the frozen expression space (`shift_source`).
- `S3` (two reference donors) is **insufficient**, however well its cells are labelled.
- `S4` has no query cells: **absent**.

The rules are evaluated in order; see the [transport guide](../transport.md).

## Cells

Each query cell is `supported`, `uncertain` or `out_of_reference`:

In [6]:
pd.crosstab(query_adata.obs["cell_state"], route.cells["category"])

category,out_of_reference,supported
cell_state,,
S0,0,305
S1,289,6
S2,5,286
S3,0,309


## Gate downstream analyses

Only `supported` states enter group-level analyses by default. Abundance keeps
out-of-reference cells in the denominator (as `unknown`), so fractions are not inflated.

In [7]:
route.supported_states

['S0', 'S2']

In [8]:
route.state_abundance(states="supported")

state,S0,S2
group,,
Q00,0.260,0.260
Q01,0.255,0.255
Q02,0.215,0.225
Q03,0.265,0.215
Q04,0.310,0.240
Q05,0.220,0.235


In [9]:
pb = route.pseudobulk(states="supported")
pb

AnnData object with n_obs × n_vars = 12 × 300
    obs: 'group', 'state', 'n_cells'
    uns: 'statetrace_pseudobulk'
    layers: None (.X)

## Another mapper

Any tool's output enters through `Predictions`. Here a logistic regression on the latent
stands in for scPoli, scANVI or Symphony (`Predictions.from_scpoli`, `from_scanvi`,
`from_symphony`, `from_celltypist` and `from_azimuth` read their usual outputs).

In [10]:
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(max_iter=1000).fit(
    ref_adata.obsm["X_latent"], ref_adata.obs["cell_state"].astype(str)
)
preds = st.Predictions.from_arrays(
    query_adata.obs_names,
    clf.predict(query_adata.obsm["X_latent"]),
    proba=clf.predict_proba(query_adata.obsm["X_latent"]),
    states=list(clf.classes_),
    backend="logistic_regression",
)
route_lr = st.validate_transport(ref, query_adata, group_key="donor_id", predictions=preds)
print(st.compare_backends({"knn": route, "logistic_regression": route_lr}).summary())

backends: knn, logistic_regression
  all-backend cell agreement 1.000
  per-state concordance: S0=1.00, S1=1.00, S2=1.00, S3=1.00
  transport status agreement across backends 1.000


The status should not depend on the mapper; `compare_backends` reports where it does.

## A population the reference never saw

A withheld state is the hard case: a mapper assigns its cells to the nearest known state.

In [11]:
ref_b, query_b, truth_b = simulate_transport(
    n_ref_groups=8,
    n_query_groups=6,
    n_states=6,
    n_latent=10,
    n_genes=300,
    novel_states=["S5"],
    shifted_states={"S1": 9.0},
    seed=5,
)
ref_b = st.Reference(
    ref_b, group_key="donor_id", state_key="cell_state", latent_key="X_latent"
).fit()
ref_b.leave_one_group_out()
ref_b = st.Reference.load(ref_b.freeze(workdir / "reference_b.statetrace")).fit()
route_b = st.validate_transport(ref_b, query_b, group_key="donor_id")
route_b.states[
    ["groups_recovered", "shift_level", "novel_population", "status", "caution_reasons"]
].join(truth_b["truth"])

,groups_recovered,shift_level,novel_population,status,caution_reasons,truth
state,,,,,,
S0,0,high,True,supported_with_caution,novel_population;recovery,stable
S1,0,high,True,shifted,,shifted
S2,6,low,False,supported,,stable
S3,6,low,False,supported,,stable
S4,6,high,False,supported_with_caution,novel_population,stable


In [12]:
pd.crosstab(query_b.obs["cell_state"], route_b.cells["category"])

category,out_of_reference,supported
cell_state,,
S0,0,213
S1,203,0
S2,1,183
S3,0,201
S4,1,199
S5,199,0


Which reference state did the mapper give them?

In [13]:
pd.crosstab(query_b.obs["cell_state"], route_b.cells["state"])

state,S0,S1,S2,S3,S4
cell_state,,,,,
S0,213,0,0,0,0
S1,0,200,0,0,3
S2,0,0,184,0,0
S3,0,0,0,201,0
S4,0,0,0,0,200
S5,188,3,0,6,2


The kNN baseline gave most withheld `S5` cells the label `S0`. StateTrace marks them out of
reference, and `S0`, which absorbed them, carries a novel population: it is kept out of
`supported` and its `caution_reasons` say why. The `S1` cells keep their label but sit far
from the reference `S1`, so that state is `shifted`. `S4` receives a caution from the
handful of such cells it absorbed.

## Report

`report` writes the state, group and cell tables with a Markdown summary; `to_obs` writes the
per-cell results into the query's `obs`.

In [14]:
out = route.report(workdir / "report")
print((out / "report.md").read_text()[:1500])

# StateTrace transport report

Backend: **knn** | query groups (`donor_id`): 6 | query cells: 1200 | reference states: 5 | created 2026-10-15T14:23:32+00:00 | StateTrace 0.1.0

## Transport status

**supported** 2, **supported_with_caution** 0, **shifted** 1, **insufficient** 1, **absent** 1

States marked `supported` have enough group-level evidence to enter downstream differential-abundance, pseudobulk or genetic analyses; `supported_with_caution` states should be reported with their caveat; `shifted`, `insufficient` and `absent` states should not be analysed as the reference construct.

## Evidence per state

| state | ref_groups | ref_reproducibility | groups_recovered | recovery_rate | median_confidence | uncertain_fraction | ood_fraction | shift_raw/residual | shift_level | shift_source | groups_shifted | novel_population | status | caution_reasons |
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
| S0 | 8 | stable | 6/6 | 1.00 | 1.00 | 0.00 | 0.00 | 6.5/-0.4 | low 

In [15]:
route.to_obs()
query_adata.obs.filter(like="statetrace").head()

,statetrace_state,statetrace_confidence,statetrace_margin,statetrace_support,statetrace_novelty,statetrace_category
cell,,,,,,
cell0,S2,1.0,1.0,1.0,-0.413839,supported
cell1,S2,1.0,1.0,1.0,0.170246,supported
cell2,S0,1.0,1.0,1.0,-0.284788,supported
cell3,S1,1.0,1.0,1.0,8.006863,out_of_reference
cell4,S0,1.0,1.0,1.0,-0.921898,supported
